In [2]:
import numpy as np
import pandas as pd
import tensorflow
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.interpolate import griddata
import unittest
import math
from scipy.stats import mvn

# 1. Import raw data

In [3]:
df = pd.read_csv(r"C:\Users\victo\Downloads\spy_2020_2022.csv", low_memory=False)

# 2. Clean data

In [4]:
df.columns = df.columns.str.strip().str.strip('[]')
df = df.drop(['QUOTE_UNIXTIME', 'QUOTE_READTIME', 'QUOTE_TIME_HOURS', 'EXPIRE_UNIX', 'C_DELTA', 'C_GAMMA', 'C_VEGA', 'C_THETA', 'C_RHO', 'P_DELTA', 'P_GAMMA', 'P_VEGA', 'P_THETA', 'P_RHO'], axis=1)
df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)
df['EXPIRE_DATE'] = pd.to_datetime(df['EXPIRE_DATE'], errors='coerce')
df['QUOTE_DATE'] = pd.to_datetime(df['QUOTE_DATE'], errors='coerce')
date_cols = ['EXPIRE_DATE', 'QUOTE_DATE']
cols_to_exclude = ['EXPIRE_DATE', 'QUOTE_DATE', 'C_SIZE', 'P_SIZE']
num_cols = df.columns.difference(cols_to_exclude)
df[date_cols] = df[date_cols].apply(pd.to_datetime, format='%Y-%m-%d', errors='coerce')
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df = df.drop(['C_IV', 'C_VOLUME', 'C_LAST', 'C_SIZE', 'C_BID', 'C_ASK', 'STRIKE_DISTANCE', 'STRIKE_DISTANCE_PCT', 'EXPIRE_DATE'], axis=1)
df['moneyness'] = df['UNDERLYING_LAST'] / df['STRIKE']
df = df[df['moneyness'] > 0.5]  # filter out moneyness <= 0.5
df = df[df['moneyness'] < 1.5]  # filter out moneyness >= 1.5
df = df[df['P_BID'] > 0]  # filter out 0 bid options
df['midprice'] = (df['P_BID'] + df['P_ASK']) / 2.0
df = df[df['midprice'] > 1] 
df = df[df['DTE'] > 0]  # filter out 0DTE options
df = df[df['P_VOLUME'] > 0] # filter out 0 volume options
df = df[df['P_IV'] < 3] # filter out extreme IV values
df = df[df['P_IV'] > 0] # filter out extreme IV values
df['T'] = df['DTE'] / 365.0

C:\Users\victo\AppData\Local\Temp\ipykernel_18304\2044094298.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)


# 3. Add additional data and clean

In [5]:
df['log_moneyness'] = np.log(df['moneyness'])
df['q'] = 0.015  # constant dividend yield
rf = pd.read_csv(r"C:\Users\victo\Downloads\DTB3.csv", low_memory=False)
rf['observation_date'] = pd.to_datetime(rf['observation_date'], format='%Y-%m-%d', errors='coerce')
rf.rename(columns={'observation_date': 'date', 'DTB3': 'risk_free_rate'}, inplace=True)
rf['risk_free_rate'] /= 100
rf.sort_values('date', inplace=True)
rf.fillna(method='ffill', inplace=True)
df.rename(columns={'QUOTE_DATE': 'date'}, inplace=True)
df = df.merge(rf, on='date', how='left')

C:\Users\victo\AppData\Local\Temp\ipykernel_18304\3430148834.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  rf.fillna(method='ffill', inplace=True)


# 4. Define Black-Scholes implied volatility

In [6]:
from scipy.optimize import brentq, newton
def black_scholes_put_price(S, K, T, r, sigma):
    """Black-Scholes formula for European put option."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price

def implied_volatility_put(market_price, S, K, T, r, 
                           initial_guess=0.2, 
                           tol=1e-6, 
                           max_iter=100):
    """Compute implied volatility using the Newton-Raphson method, with fallback."""
    
    def objective(sigma):
        return black_scholes_put_price(S, K, T, r, sigma) - market_price

    try:
        # Try Newton-Raphson with derivative
        iv = newton(objective, initial_guess, tol=tol, maxiter=max_iter)
    except (RuntimeError, OverflowError):
        # Fallback to Brent's method if Newton-Raphson fails
        iv = brentq(objective, 1e-6, 5.0, xtol=tol)
    return iv

# 5. Get correct BS implied vol from market prices

In [7]:
def compute_iv(row):
    try:
        iv = implied_volatility_put(row['midprice'], row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], initial_guess=0.2, tol=1e-3, max_iter=100)
    except Exception as e:
        iv = np.nan
    return iv
# df['implied_volatility'] = df.apply(compute_iv, axis=1)

In [8]:
# df must contain: a Date column and a Price column
# 1.  Make sure the date column is a datetime index with one row per date
vol_df = (
    df                                   # your original dataframe
    .assign(Date=pd.to_datetime(df['date']))
    .groupby('date', as_index=False)     # collapse any duplicate rows per date
    .agg({'UNDERLYING_LAST': 'first'})             # or 'mean' / 'last' – your choice
    .sort_values('date')
    .set_index('date')
)

# 2.  Compute daily log-returns
vol_df['log_ret'] = np.log(vol_df['UNDERLYING_LAST'] / vol_df['UNDERLYING_LAST'].shift(1))

# 3.  20-day rolling *daily* volatility (σ20)
vol_df['vol_20d'] = vol_df['log_ret'].rolling(window=20).std(ddof=0)      # population stdev

# 4.  Scale to annual volatility  (√252 ≈ trading days in a year)
vol_df['vol_20d_annual'] = vol_df['vol_20d'] * np.sqrt(252)




In [9]:
vol_df = vol_df.reset_index().rename(columns={'index': 'date'})
df = df.merge(vol_df[['date', 'vol_20d_annual']], on='date', how='left')


In [10]:
df['vol_20d_annual'].fillna(0.20, inplace=True)

C:\Users\victo\AppData\Local\Temp\ipykernel_18304\713945868.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['vol_20d_annual'].fillna(0.20, inplace=True)


In [11]:
df.sort_values(by=['vol_20d_annual'], ascending=True, inplace=True)
df.head()

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q,risk_free_rate,vol_20d_annual
1161358,2021-11-24,469.44,113.96,406.0,5.64,5.68,99 x 143,5.74,0.26286,49.0,1.156256,5.660,0.312219,0.145187,0.015,0.0006,0.062803
1160919,2021-11-24,469.44,28.00,465.0,6.13,6.29,78 x 48,6.25,0.14928,67.0,1.009548,6.210,0.076712,0.009503,0.015,0.0006,0.062803
1160918,2021-11-24,469.44,28.00,464.0,5.90,5.97,25 x 25,6.39,0.15215,113.0,1.011724,5.935,0.076712,0.011656,0.015,0.0006,0.062803
1160917,2021-11-24,469.44,28.00,462.0,5.37,5.42,23 x 23,5.74,0.15793,4.0,1.016104,5.395,0.076712,0.015976,0.015,0.0006,0.062803
1160916,2021-11-24,469.44,28.00,460.0,4.91,4.96,27 x 27,5.27,0.16364,25.0,1.020522,4.935,0.076712,0.020314,0.015,0.0006,0.062803


# 6. Split data

In [12]:
# Sort by date
df = df.sort_values('date')

# Determine cutoff date (e.g., 80% point)
cutoff_date = df['date'].quantile(0.8)

# Create splits
train_df = df[df['date'] <= cutoff_date]
test_df = df[df['date'] > cutoff_date]

# 7. Define Bjerklund-Stensland model

In [13]:
def _phi(fs, t, gamma, h, i, r, b, v):
    d1 = -(math.log(fs / h) + (b + (gamma - 0.5) * (v ** 2)) * t) / (v * math.sqrt(t))
    d2 = d1 - 2 * math.log(i / fs) / (v * math.sqrt(t))

    lambda1 = (-r + gamma * b + 0.5 * gamma * (gamma - 1) * (v ** 2))
    kappa = (2 * b) / (v ** 2) + (2 * gamma - 1)

    phi = math.exp(lambda1 * t) * (fs ** gamma) * (norm.cdf(d1) - ((i / fs) ** kappa) * norm.cdf(d2))
    return phi

def _cbnd(a, b, rho):
    # This distribution uses the Genz multi-variate normal distribution 
    # code found as part of the standard SciPy distribution
    lower = np.array([0, 0])
    upper = np.array([a, b])
    infin = np.array([0, 0])
    correl = rho
    error, value, inform = mvn.mvndst(lower, upper, infin, correl)
    return value


def _psi(fs, t2, gamma, h, i2, i1, t1, r, b, v):
    vsqrt_t1 = v * math.sqrt(t1)
    vsqrt_t2 = v * math.sqrt(t2)

    bgamma_t1 = (b + (gamma - 0.5) * (v ** 2)) * t1
    bgamma_t2 = (b + (gamma - 0.5) * (v ** 2)) * t2

    d1 = (math.log(fs / i1) + bgamma_t1) / vsqrt_t1
    d3 = (math.log(fs / i1) - bgamma_t1) / vsqrt_t1

    d2 = (math.log((i2 ** 2) / (fs * i1)) + bgamma_t1) / vsqrt_t1
    d4 = (math.log((i2 ** 2) / (fs * i1)) - bgamma_t1) / vsqrt_t1

    e1 = (math.log(fs / h) + bgamma_t2) / vsqrt_t2
    e2 = (math.log((i2 ** 2) / (fs * h)) + bgamma_t2) / vsqrt_t2
    e3 = (math.log((i1 ** 2) / (fs * h)) + bgamma_t2) / vsqrt_t2
    e4 = (math.log((fs * (i1 ** 2)) / (h * (i2 ** 2))) + bgamma_t2) / vsqrt_t2

    tau = math.sqrt(t1 / t2)
    lambda1 = (-r + gamma * b + 0.5 * gamma * (gamma - 1) * (v ** 2))
    kappa = (2 * b) / (v ** 2) + (2 * gamma - 1)

    psi = math.exp(lambda1 * t2) * (fs ** gamma) * (_cbnd(-d1, -e1, tau)
                                                    - ((i2 / fs) ** kappa) * _cbnd(-d2, -e2, tau)
                                                    - ((i1 / fs) ** kappa) * _cbnd(-d3, -e3, -tau)
                                                    + ((i1 / i2) ** kappa) * _cbnd(-d4, -e4, -tau))
    return psi


def _bjerksund_stensland_2002(fs, x, t, r, b, v):
    # preliminary calculations
    v2 = v ** 2
    t1 = 0.5 * (math.sqrt(5) - 1) * t
    t2 = t

    beta_inside = ((b / v2 - 0.5) ** 2) + 2 * r / v2
    # forcing the inside of the sqrt to be a positive number
    beta_inside = abs(beta_inside)
    beta = (0.5 - b / v2) + math.sqrt(beta_inside)
    b_infinity = (beta / (beta - 1)) * x
    b_zero = max(x, (r / (r - b)) * x)

    h1 = -(b * t1 + 2 * v * math.sqrt(t1)) * ((x ** 2) / ((b_infinity - b_zero) * b_zero))
    h2 = -(b * t2 + 2 * v * math.sqrt(t2)) * ((x ** 2) / ((b_infinity - b_zero) * b_zero))

    i1 = b_zero + (b_infinity - b_zero) * (1 - math.exp(h1))
    i2 = b_zero + (b_infinity - b_zero) * (1 - math.exp(h2))

    alpha1 = (i1 - x) * (i1 ** (-beta))
    alpha2 = (i2 - x) * (i2 ** (-beta))

    # check for immediate exercise
    if fs >= i2:
        value = fs - x
    else:
        # Perform the main calculation    
        value = (alpha2 * (fs ** beta)
                 - alpha2 * _phi(fs, t1, beta, i2, i2, r, b, v)
                 + _phi(fs, t1, 1, i2, i2, r, b, v)
                 - _phi(fs, t1, 1, i1, i2, r, b, v)
                 - x * _phi(fs, t1, 0, i2, i2, r, b, v)
                 + x * _phi(fs, t1, 0, i1, i2, r, b, v)
                 + alpha1 * _phi(fs, t1, beta, i1, i2, r, b, v)
                 - alpha1 * _psi(fs, t2, beta, i1, i2, i1, t1, r, b, v)
                 + _psi(fs, t2, 1, i1, i2, i1, t1, r, b, v)
                 - _psi(fs, t2, 1, x, i2, i1, t1, r, b, v)
                 - x * _psi(fs, t2, 0, i1, i2, i1, t1, r, b, v)
                 + x * _psi(fs, t2, 0, x, i2, i1, t1, r, b, v))

        
    return value

def BS2002(type, fs, x, t, r, b, v):
    if type == 'c':
        return _bjerksund_stensland_2002(fs, x, t, r, b, v)
    elif type == 'p':
        put__x = fs
        put_fs = x
        put_b = -b
        put_r = r - b
        return _bjerksund_stensland_2002(put_fs, put__x, t, put_r, put_b, v)

# Longstaff-Schwartz

In [15]:
import numpy as np

def price_american_put_LSM(S0, K, T, r, q, sigma, num_paths, num_steps):
    """
    Price an American put option using Longstaff-Schwartz Least Squares Monte Carlo.
    
    Parameters:
        S0 (float)      : Initial stock price
        K (float)       : Strike price of the put
        T (float)       : Time to expiration in years
        r (float)       : Risk-free interest rate (annual continuously compounded)
        q (float)       : Continuous dividend yield of the stock
        sigma (float)   : Volatility of the stock (annual standard deviation)
        num_paths (int) : Number of Monte Carlo simulation paths
        num_steps (int) : Number of time steps (discrete exercise opportunities)
    
    Returns:
        float: Estimated American put option price.
    """
    # Time increment
    dt = T / num_steps
    # Pre-compute drift and volatility factors for efficiency
    drift = (r - q - 0.5 * sigma**2) * dt
    vol = sigma * np.sqrt(dt)
    
    # 1. Simulate risk-neutral GBM paths for the underlying stock
    # Initialize array for stock paths: dimensions (num_steps+1) x (num_paths)
    S_paths = np.empty((num_steps+1, num_paths))
    S_paths[0, :] = S0
    # Generate all random shocks at once for efficiency
    # Each step has num_paths independent N(0,1) draws
    Z = np.random.normal(size=(num_steps, num_paths))
    for t in range(1, num_steps+1):
        # Vectorized stock price update: S[t] = S[t-1] * exp(drift + vol * Z)
        S_paths[t, :] = S_paths[t-1, :] * np.exp(drift + vol * Z[t-1, :])
    
    # 2. Compute payoff at maturity for all paths (time index num_steps)
    # For a put option, payoff = max(K - S, 0)
    payoffs = np.maximum(K - S_paths[num_steps, :], 0.0)
    
    # 3. Backward induction to incorporate early exercise opportunities
    # We'll use an array `cashflows` to track the optimal payoff (discounted) from each path
    cashflows = payoffs.copy()  # start with terminal payoff at T for each path
    # Work backward from second-last time step down to first time step
    for t in range(num_steps-1, 0, -1):
        # Discount the cashflows from the next time step back to time t
        cashflows *= np.exp(-r * dt)
        # Get stock prices at time t
        S_t = S_paths[t, :]
        # Identify paths where option is in the money at time t (put: S_t < K)
        in_the_money = S_t < K
        if not np.any(in_the_money):
            # If no path is in the money, move to the previous time step
            continue
        
        # Prepare regression to estimate continuation value: 
        X = S_t[in_the_money]  # stock prices for in-the-money paths
        Y = cashflows[in_the_money]  # discounted future cashflows for those paths
        
        # Construct Laguerre polynomial basis:
        # We'll use [1, L1(x), L2(x)] where L1(x) = 1 - x, L2(x) = 1 - 2x + 0.5*x^2.
        # (Optionally, x could be scaled like x = S_t/K to improve numerical stability)
        x = X  # using X directly; could also use X/K for normalization
        L0 = np.ones_like(x)
        L1 = 1.0 - x
        L2 = 1.0 - 2.0*x + 0.5 * x**2
        # Design matrix for regression (shape: n_paths_in_money x 3)
        A = np.vstack([L0, L1, L2]).T
        
        # Solve least-squares regression A * beta ≈ Y to get regression coefficients
        beta, *_ = np.linalg.lstsq(A, Y, rcond=None)
        # Estimated continuation value for all in-the-money paths at time t
        C_hat = beta[0] + beta[1]*(1.0 - S_t) + beta[2]*(1.0 - 2.0*S_t + 0.5*S_t**2)
        
        # Decide whether to exercise or continue for in-the-money paths
        exercise_value = np.maximum(K - S_t, 0.0)  # immediate payoff if exercised at t
        # If exercise value is greater than continuation estimate, exercise (replace cashflow)
        exercise_indices = in_the_money & (exercise_value > C_hat)
        cashflows[exercise_indices] = exercise_value[exercise_indices]
        # Paths not meeting the condition will keep their future cashflow (already in `cashflows`)
    # end for loop
    
    # 4. At time 0, the price is the average of discounted cashflows (already at t=0 after loop)
    option_price = np.exp(-r * dt) * np.mean(cashflows)  # discount from t=1 to t=0 and average
    return option_price


In [22]:
price_american_put_LSM(S0=40, K=40, T=1.0, r=0.06, q=0, sigma=0.2, num_paths=10000, num_steps=1000)

2.324936760328773

In [27]:
target_date = pd.to_datetime('2021-09-01')
LSM_df = df[df['date'] == target_date].copy()

LSM_df['LSM_price_const_vol'] = LSM_df.apply(lambda row: price_american_put_LSM(row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], 0.015, 0.2, num_paths=10000, num_steps=1000), axis=1)


In [28]:
LSM_df.head()

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q,risk_free_rate,vol_20d_annual,LSM_price_const_vol
761,2021-09-01,451.85,79.04,374.0,2.03,2.05,1065 x 235,2.12,0.28702,3.0,1.208155,2.040,0.216548,0.189094,0.015,0.0005,0.076577,0.348411
759,2021-09-01,451.85,79.04,372.0,1.96,1.97,105 x 204,1.91,0.29063,27.0,1.214651,1.965,0.216548,0.194456,0.015,0.0005,0.076577,0.294452
758,2021-09-01,451.85,79.04,370.0,1.88,1.89,636 x 180,1.92,0.29412,123.0,1.221216,1.885,0.216548,0.199847,0.015,0.0005,0.076577,0.242245
757,2021-09-01,451.85,79.04,369.0,1.85,1.86,50 x 405,1.84,0.29628,1.0,1.224526,1.855,0.216548,0.202554,0.015,0.0005,0.076577,0.201454
773,2021-09-01,451.85,79.04,388.0,2.71,2.72,229 x 91,2.61,0.26227,9.0,1.164562,2.715,0.216548,0.152345,0.015,0.0005,0.076577,0.915548


In [29]:
# Errors
abs_error_sum_const_vol = 0
rmse_sum_const_vol = 0
rel_error_sum_const_vol = 0
for i in range(len(LSM_df)):
    abs_error_sum_const_vol += np.abs(LSM_df['LSM_price_const_vol'].iloc[i] - LSM_df['midprice'].iloc[i])
    rmse_sum_const_vol += (LSM_df['LSM_price_const_vol'].iloc[i] - LSM_df['midprice'].iloc[i])**2
    rel_error_sum_const_vol += np.abs((LSM_df['LSM_price_const_vol'].iloc[i] - LSM_df['midprice'].iloc[i]) / np.abs(LSM_df['midprice'].iloc[i]))

# Metrics
mae_const_vol = abs_error_sum_const_vol / len(LSM_df)
rmse_const_vol = np.sqrt(rmse_sum_const_vol / len(LSM_df))
rel_error_const_vol = rel_error_sum_const_vol / len(LSM_df)
re_sum_const_vol = rel_error_sum_const_vol


print(f"MAE: {mae_const_vol:.4f}")
print(f"RMSE: {rmse_const_vol:.4f}")
print(f"Sum of RE: {re_sum_const_vol:.4f}")
print(f"Mean RE: {rel_error_const_vol:.4f}")

MAE: 2.8910
RMSE: 3.6690
Sum of RE: 759.9291
Mean RE: 0.3975


# 8. Apply Bjerksund-Stensland model

In [13]:
test_df['Bjerksund_Stensland_price_const_vol'] = test_df.apply(lambda row: BS2002('p', row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], 0.015, 0.2), axis=1)
train_df['Bjerksund_Stensland_price_const_vol'] = train_df.apply(lambda row: BS2002('p', row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], 0.015, 0.2), axis=1)
test_df['Bjerksund_Stensland_price_rolling_vol'] = test_df.apply(lambda row: BS2002('p', row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], 0.015, row['vol_20d_annual']), axis=1)
train_df['Bjerksund_Stensland_price_rolling_vol'] = train_df.apply(lambda row: BS2002('p', row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], 0.015, row['vol_20d_annual']), axis=1)

C:\Users\victo\AppData\Local\Temp\ipykernel_14136\2514774326.py:18: DeprecationWarning: `scipy.stats.mvn.mvndst` is deprecated along with the `scipy.stats.mvn` namespace. `scipy.stats.mvn.mvndst` will be removed in SciPy 1.14.0, and the `scipy.stats.mvn` namespace will be removed in SciPy 2.0.0.
  error, value, inform = mvn.mvndst(lower, upper, infin, correl)


KeyboardInterrupt: 

# 9. Calculate error metrics

In [31]:
df = df.dropna()
test_df = test_df.dropna()
train_df = train_df.dropna()

## Constant vol

In [ ]:
# Errors
abs_error_sum_const_vol = 0
rmse_sum_const_vol = 0
rel_error_sum_const_vol = 0
for i in range(len(test_df)):
    abs_error_sum_const_vol += np.abs(test_df['Bjerksund_Stensland_price_const_vol'].iloc[i] - test_df['midprice'].iloc[i])
    rmse_sum_const_vol += (test_df['Bjerksund_Stensland_price_const_vol'].iloc[i] - test_df['midprice'].iloc[i])**2
    rel_error_sum_const_vol += np.abs((test_df['Bjerksund_Stensland_price_const_vol'].iloc[i] - test_df['midprice'].iloc[i]) / np.abs(test_df['midprice'].iloc[i]))

# Metrics
mae_const_vol = abs_error_sum_const_vol / len(test_df)
rmse_const_vol = np.sqrt(rmse_sum_const_vol / len(test_df))
rel_error_const_vol = rel_error_sum_const_vol / len(test_df)
re_sum_const_vol = rel_error_sum_const_vol


print(f"MAE: {mae_const_vol:.4f}")
print(f"RMSE: {rmse_const_vol:.4f}")
print(f"Sum of RE: {re_sum_const_vol:.4f}")
print(f"Mean RE: {rel_error_const_vol:.4f}")

MAE: 2.6411
RMSE: 3.5058
Sum of RE: 100589.6444
Mean RE: 0.3626


In [ ]:
# Errors
abs_error_sum_const_vol = 0
rmse_sum_const_vol = 0
rel_error_sum_const_vol = 0
for i in range(len(train_df)):
    abs_error_sum_const_vol += np.abs(train_df['Bjerksund_Stensland_price_const_vol'].iloc[i] - train_df['midprice'].iloc[i])
    rmse_sum_const_vol += (train_df['Bjerksund_Stensland_price_const_vol'].iloc[i] - train_df['midprice'].iloc[i])**2
    rel_error_sum_const_vol += np.abs((train_df['Bjerksund_Stensland_price_const_vol'].iloc[i] - train_df['midprice'].iloc[i]) / np.abs(train_df['midprice'].iloc[i]))

# Metrics
mae_const_vol = abs_error_sum_const_vol / len(train_df)
rmse_const_vol = np.sqrt(rmse_sum_const_vol / len(train_df))
rel_error_const_vol = rel_error_sum_const_vol / len(train_df)
re_sum_const_vol = rel_error_sum_const_vol


print(f"MAE: {mae_const_vol:.4f}")
print(f"RMSE: {rmse_const_vol:.4f}")
print(f"Sum of RE: {re_sum_const_vol:.4f}")
print(f"Mean RE: {rel_error_const_vol:.4f}")

MAE: 3.7299
RMSE: 5.0374
Sum of RE: 438023.5245
Mean RE: 0.3939


## Rolling vol

In [ ]:
# Errors
abs_error_sum_rolling_vol = 0
rmse_sum_rolling_vol = 0
rel_error_sum_rolling_vol = 0
for i in range(len(train_df)):
    abs_error_sum_rolling_vol += np.abs(train_df['Bjerksund_Stensland_price_rolling_vol'].iloc[i] - train_df['midprice'].iloc[i])
    rmse_sum_rolling_vol += (train_df['Bjerksund_Stensland_price_rolling_vol'].iloc[i] - train_df['midprice'].iloc[i])**2
    rel_error_sum_rolling_vol += np.abs((train_df['Bjerksund_Stensland_price_rolling_vol'].iloc[i] - train_df['midprice'].iloc[i]) / np.abs(train_df['midprice'].iloc[i]))

# Metrics
mae_rolling_vol = abs_error_sum_rolling_vol / len(train_df)
rmse_rolling_vol = np.sqrt(rmse_sum_rolling_vol / len(train_df))
rel_error_rolling_vol = rel_error_sum_rolling_vol / len(train_df)
re_sum_rolling_vol = rel_error_sum_rolling_vol


print(f"MAE: {mae_rolling_vol:.4f}")
print(f"RMSE: {rmse_rolling_vol:.4f}")
print(f"Sum of RE: {re_sum_rolling_vol:.4f}")
print(f"Mean RE: {rel_error_rolling_vol:.4f}")

MAE: 6.5377
RMSE: 11.0808
Sum of RE: 589477.0808
Mean RE: 0.5302


In [ ]:
# Errors
abs_error_sum_rolling_vol = 0
rmse_sum_rolling_vol = 0
rel_error_sum_rolling_vol = 0
for i in range(len(test_df)):
    abs_error_sum_rolling_vol += np.abs(test_df['Bjerksund_Stensland_price_rolling_vol'].iloc[i] - test_df['midprice'].iloc[i])
    rmse_sum_rolling_vol += (test_df['Bjerksund_Stensland_price_rolling_vol'].iloc[i] - test_df['midprice'].iloc[i])**2
    rel_error_sum_rolling_vol += np.abs((test_df['Bjerksund_Stensland_price_rolling_vol'].iloc[i] - test_df['midprice'].iloc[i]) / np.abs(test_df['midprice'].iloc[i]))

# Metrics
mae_rolling_vol = abs_error_sum_rolling_vol / len(test_df)
rmse_rolling_vol = np.sqrt(rmse_sum_rolling_vol / len(test_df))
rel_error_rolling_vol = rel_error_sum_rolling_vol / len(test_df)
re_sum_rolling_vol = rel_error_sum_rolling_vol


print(f"MAE: {mae_rolling_vol:.4f}")
print(f"RMSE: {rmse_rolling_vol:.4f}")
print(f"Sum of RE: {re_sum_rolling_vol:.4f}")
print(f"Mean RE: {rel_error_rolling_vol:.4f}")

MAE: 2.2950
RMSE: 3.4989
Sum of RE: 78974.3192
Mean RE: 0.2846


In [ ]:
train_df.head()

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q,risk_free_rate,vol_20d_annual,Bjerksund_Stensland_price_const_vol,Bjerksund_Stensland_price_rolling_vol
921580,2020-01-02,324.87,1.0,326.5,1.83,1.91,42 x 50,1.97,0.08399,104.0,0.995008,1.870,0.002740,-0.005005,0.015,0.0151,0.2,2.319107,2.319107
921930,2020-01-02,324.87,50.0,335.0,10.75,10.82,40 x 29,11.76,0.07651,275.0,0.969761,10.785,0.136986,-0.030705,0.015,0.0151,0.2,15.230001,15.230001
921931,2020-01-02,324.87,50.0,336.0,11.64,11.76,54 x 200,12.63,0.07776,36.0,0.966875,11.700,0.136986,-0.033686,0.015,0.0151,0.2,15.904776,15.904776
921932,2020-01-02,324.87,50.0,338.0,13.39,13.55,200 x 200,17.14,0.07686,2.0,0.961154,13.470,0.136986,-0.039621,0.015,0.0151,0.2,17.297904,17.297904
921933,2020-01-02,324.87,50.0,339.0,14.34,14.50,200 x 200,15.27,0.07833,5.0,0.958319,14.420,0.136986,-0.042575,0.015,0.0151,0.2,18.015618,18.015618


# 10. Define Neural Network

In [36]:
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader


class OptionpricingDataset(Dataset):
    def __init__(self, df, scaler=None, fit_scaler=False):
        df = df.copy()
        self.scaler = scaler
        self.features_cols = ['log_moneyness', 'T', 'risk_free_rate', 'q', 'UNDERLYING_LAST', 'STRIKE', 'P_VOLUME', 'vol_20d_annual']
        self.target_col = ['midprice']
        X = df[self.features_cols].values.astype(np.float32)
        y = df[self.target_col].values.astype(np.float32).reshape(-1, 1)

        # apply scaling
        if scaler is None:
            self.scaler = StandardScaler()
            self.X = self.scaler.fit_transform(X)
        else:
            self.scaler = scaler
            if fit_scaler:
                self.X = self.scaler.fit_transform(X)
            else:
                self.X = self.scaler.transform(X)

        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = OptionpricingDataset(train_df, fit_scaler=True)
scaler = train_dataset.scaler  # Save the fitted scaler

test_dataset = OptionpricingDataset(test_df, scaler=scaler, fit_scaler=False)

import torch.nn as nn

class OptionPricingNN(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[40,40,40,40,]):
        super(OptionPricingNN, self).__init__()
        layers = []
        in_dim = input_dim

        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, 1))  # Output layer
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

def train_model(model, train_dataset, test_dataset, epochs=10, batch_size=128, lr=1e-3):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    def mape_loss(y_pred, y_true, eps=1e-6):
        return torch.mean(torch.abs((y_true - y_pred) / (y_true + eps)))
    class RMSELoss(nn.Module):
        def __init__(self, eps=1e-6):
            super(RMSELoss, self).__init__()
            self.mse = nn.MSELoss()
            self.eps = eps

        def forward(self, y_pred, y_true):
            return torch.sqrt(self.mse(y_pred, y_true) + self.eps)
     #criterion = nn.SmoothL1Loss()  # Use SmoothL1Loss for regression
    criterion = RMSELoss()  # Use RMSE loss for regression

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    def lr_lambda(epoch):
        if epoch < 20:
            return 1.0       # first 20 epochs: 1e-3*1.0 = 1e-3
        elif epoch < 60:
            return 0.1       # next 20 epochs: 1e-3*0.1 = 1e-4
        else:
            return 0.01      # subsequent epochs: 1e-3*0.01 = 1e-5 (example)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
    
        scheduler.step()  # update the learning rate based on the epoch

        avg_train_loss = train_loss / len(train_dataset)

        # Evaluation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss += loss.item() * X_batch.size(0)

        avg_test_loss = test_loss / len(test_dataset)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.3f} | Test Loss: {avg_test_loss:.3f}")

    return model

In [39]:
input_dim = len(train_dataset.features_cols)
model = OptionPricingNN(input_dim=input_dim)

# Train model
trained_model = train_model(model, train_dataset, test_dataset, epochs=80)

Epoch 1/40 | Train Loss: 1.645 | Test Loss: 9.393
Epoch 2/40 | Train Loss: 1.074 | Test Loss: 8.991
Epoch 3/40 | Train Loss: 1.002 | Test Loss: 9.821
Epoch 4/40 | Train Loss: 0.928 | Test Loss: 7.858
Epoch 5/40 | Train Loss: 0.869 | Test Loss: 5.671
Epoch 6/40 | Train Loss: 0.837 | Test Loss: 5.429
Epoch 7/40 | Train Loss: 0.810 | Test Loss: 5.678
Epoch 8/40 | Train Loss: 0.783 | Test Loss: 6.072
Epoch 9/40 | Train Loss: 0.760 | Test Loss: 5.956
Epoch 10/40 | Train Loss: 0.740 | Test Loss: 6.032
Epoch 11/40 | Train Loss: 0.723 | Test Loss: 6.327
Epoch 12/40 | Train Loss: 0.709 | Test Loss: 5.973
Epoch 13/40 | Train Loss: 0.700 | Test Loss: 5.781
Epoch 14/40 | Train Loss: 0.690 | Test Loss: 5.545
Epoch 15/40 | Train Loss: 0.679 | Test Loss: 4.920
Epoch 16/40 | Train Loss: 0.665 | Test Loss: 4.875
Epoch 17/40 | Train Loss: 0.655 | Test Loss: 4.630
Epoch 18/40 | Train Loss: 0.645 | Test Loss: 4.739
Epoch 19/40 | Train Loss: 0.634 | Test Loss: 5.274
Epoch 20/40 | Train Loss: 0.628 | Test L

KeyboardInterrupt: 

In [ ]:
len(test_df)

277448

In [ ]:
import optuna
from torch.utils.data import DataLoader

def objective(trial):
    # Sample hyperparameters
    hidden_layer_sizes = trial.suggest_categorical("hidden_layer_sizes", [[40,40,40,40], [128,128,128], [400,400,400], [16,32,16]])
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 512, 1024, 4096, 8192])
    epochs = trial.suggest_int("epochs", 10, 50, step=10)
    
    # Build model using the sampled hidden layer sizes
    model = OptionPricingNN(input_dim=input_dim, hidden_sizes=hidden_layer_sizes)
    
    # Create dataloaders with the trial batch_size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # Use the same loss function/criterion as in your training function
    class RMSELoss(nn.Module):
        def __init__(self, eps=1e-6):
            super(RMSELoss, self).__init__()
            self.mse = nn.MSELoss()
            self.eps = eps

        def forward(self, y_pred, y_true):
            return torch.sqrt(self.mse(y_pred, y_true) + self.eps)
     #criterion = nn.SmoothL1Loss()  # Use SmoothL1Loss for regression
    criterion = RMSELoss()  # Use RMSE loss for regression  # Ensure RMSELoss is defined
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Train model for the sampled number of epochs
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
    
    # Evaluate test loss after training
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            test_loss += loss.item() * X_batch.size(0)
    
    avg_test_loss = test_loss / len(test_dataset)
    return avg_test_loss

# Run the hyperparameter search using Optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best trial:")
trial = study.best_trial
print("Test Loss:", trial.value)
print("Best hyperparameters:", trial.params)

[I 2025-05-31 19:20:35,701] A new study created in memory with name: no-name-bcf13777-d9cc-4049-8275-69ebd6e28a55
c:\Users\victo\anaconda3\Lib\site-packages\optuna\distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [40, 40, 40, 40] which is of type list.
  warnings.warn(message)
c:\Users\victo\anaconda3\Lib\site-packages\optuna\distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [128, 128, 128] which is of type list.
  warnings.warn(message)
c:\Users\victo\anaconda3\Lib\site-packages\optuna\distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [400, 400, 400] which is of type list.
  warnings.warn(message)
c:\Users\victo\anaconda3\Lib\site-packages\optuna\distrib

KeyboardInterrupt: 

In [34]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def get_prediction_table(model, test_dataset, original_df=None, num_rows=10000000):
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    X = test_dataset.X.to(device)
    y_true = test_dataset.y.cpu().numpy()
    
    with torch.no_grad():
        y_pred = model(X).cpu().numpy()

    # Compute errors
    abs_error = np.abs(y_true.flatten() - y_pred.flatten())
    rel_error = abs_error / np.maximum(np.abs(y_true.flatten()), 1e-6)

    mae = np.mean(abs_error)
    rmse = np.sqrt(np.mean((y_true.flatten() - y_pred.flatten()) ** 2))
    rel_error_sum = np.sum(rel_error)

    print(f"\nMAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"Sum of Relative Errors: {rel_error_sum:.6f}")

    # Create table
    data = {
        'Actual Price': y_true.flatten(),
        'Predicted Price': y_pred.flatten(),
        'Absolute Error': abs_error,
        'Relative Error': rel_error
    }

    df_result = pd.DataFrame(data)

    # Optionally join original metadata
    if original_df is not None:
        original_df = original_df.reset_index(drop=True)
        df_result = pd.concat([original_df.reset_index(drop=True), df_result], axis=1)

    return df_result.head(num_rows)

prediction_table = get_prediction_table(trained_model, test_dataset, original_df=test_df)




MAE: 3.456575
RMSE: 5.062739
Sum of Relative Errors: 107295.109375


In [ ]:
prediction_table.sort_values(by=['Relative Error'], ascending=False, inplace=True)

In [ ]:
prediction_table.sort_values(by=['Absolute Error'], ascending=False, inplace=True)
prediction_table.head(n=10)

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,...,T,log_moneyness,q,risk_free_rate,implied_volatility,Bjerksund_Stensland_price,Actual Price,Predicted Price,Absolute Error,Relative Error
170047,2022-09-05,392.24,11.0,700.0,307.82,309.06,4 x 4,273.76,1.61142,5.0,...,0.030137,-0.579206,0.015,0.0285,1.666867,307.76,1.666867,10.034372,8.367506,5.019901
100970,2022-06-14,373.83,3.0,685.0,312.30,313.54,2 x 2,311.34,0.00109,1.0,...,0.008219,-0.605618,0.015,0.0178,3.543247,311.17,3.543247,11.747478,8.204231,2.315456
116408,2022-07-06,383.31,9.0,650.0,266.59,267.15,59 x 40,264.60,2.15833,1.0,...,0.024658,-0.528128,0.015,0.0185,1.486744,266.69,1.486744,9.510008,8.023264,5.396535
113844,2022-07-04,381.24,11.0,650.0,268.71,269.33,60 x 18,281.60,1.99663,1.0,...,0.030137,-0.533543,0.015,0.0166,1.395420,268.76,1.395420,9.049053,7.653633,5.484825
102447,2022-06-15,379.15,2.0,685.0,306.80,308.05,1 x 1,308.07,0.00072,1.0,...,0.005479,-0.591487,0.015,0.0169,4.163850,305.85,4.163850,11.584944,7.421094,1.782267
102448,2022-06-15,379.15,2.0,680.0,301.80,303.05,1 x 1,303.05,0.00110,1.0,...,0.005479,-0.584161,0.015,0.0169,4.123011,300.85,4.123011,11.429621,7.306610,1.772154
113832,2022-07-04,381.24,11.0,640.0,258.73,259.33,30 x 30,198.82,1.92493,1.0,...,0.030137,-0.518039,0.015,0.0166,1.363473,258.76,1.363473,8.647522,7.284049,5.342274
176869,2022-09-12,410.94,4.0,710.0,300.00,301.19,1 x 2,301.82,2.21579,1.0,...,0.010959,-0.546818,0.015,0.0309,2.766918,299.06,2.766918,9.797419,7.030500,2.540914
86874,2022-05-30,415.26,18.0,690.0,274.98,275.74,8 x 50,232.00,1.21865,1.0,...,0.049315,-0.507787,0.015,0.0106,1.115234,274.74,1.115234,7.431940,6.316706,5.664020
115366,2022-07-05,381.95,10.0,600.0,218.19,218.74,52 x 20,218.80,1.16020,1.0,...,0.027397,-0.451640,0.015,0.0186,1.314345,218.05,1.314345,7.408576,6.094231,4.636705
